In [ ]:
-- 1. TABLA DETALLE DESCARGAS
CREATE TABLE datos_cosecha.detalle_descargas (
    id_reg VARCHAR(50) PRIMARY KEY,
    fecha_reg TIMESTAMP,
    equipo VARCHAR(100),
    cod_equipo VARCHAR(100),
    nombre_operador VARCHAR(255),
    horometro NUMERIC(10, 2),
    cantidad_litros NUMERIC(10, 2),
    tipo_carga VARCHAR(100),
    observaciones TEXT,
    tanque VARCHAR(100),
    propiedad VARCHAR(255),
    labor VARCHAR(100),
    usuario_reg VARCHAR(150),
    nombre_usuario VARCHAR(255),
    foto_horometro TEXT,
    foto_contador TEXT
);

-- 2. TABLA DETALLE MOVIMIENTOS
CREATE TABLE datos_cosecha.detalle_movimientos (
    id_reg VARCHAR(50) PRIMARY KEY,
    fecha_reg TIMESTAMP,
    tanque_origen VARCHAR(150),
    responsable_tanque_origen VARCHAR(255),
    tanque_destino VARCHAR(150),
    responsable_tanque_destino VARCHAR(255),
    cantidad_litros NUMERIC(10, 2),
    foto_contador TEXT,
    observaciones TEXT,
    usuario VARCHAR(150)
);

In [5]:
from oauth2client.service_account import ServiceAccountCredentials
import gspread
import pandas as pd

import sys
sys.path.append('..')
from sqlalchemy import create_engine, text
from sqlalchemy.dialects.postgresql import insert

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

PATH_FILE_GAPI = "pydrive-456115-66e50cca8f58.json"
POSTGRES_UTEA['DATABASE'] = 'utea_precision'

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

In [6]:
# --- Constantes globales ---
SPREADSHEET_ID = "1ti67cs_nz2XAB7JFUV6-x21pc7zo-DMxwKTijpivjKs"
WORKSHEET_DESCARGAS = "detalle_descargas"
WORKSHEET_MOVIMIENTOS = "detalle_movimientos"

# =====================================================================
# 1. FLUJO: DETALLE DESCARGAS
# =====================================================================
def obtener_cliente_sheets():
    scope = [
        "https://spreadsheets.google.com/feeds",
        'https://www.googleapis.com/auth/spreadsheets', 
        "https://www.googleapis.com/auth/drive.file", 
        "https://www.googleapis.com/auth/drive"
    ]
    creds = ServiceAccountCredentials.from_json_keyfile_name(PATH_FILE_GAPI, scope)
    return gspread.authorize(creds)

def extraer_y_filtrar_descargas(fecha_busqueda_str):
    client = obtener_cliente_sheets()
    spreadsheet = client.open_by_key(SPREADSHEET_ID)
    sheet = spreadsheet.worksheet(WORKSHEET_DESCARGAS)
    
    df = pd.DataFrame(sheet.get_all_records())
    if df.empty: return df
        
    df['fecha_reg'] = pd.to_datetime(df['fecha_reg'], format='mixed', errors='coerce')
    fecha_objetivo = pd.to_datetime(fecha_busqueda_str).date()
    return df[df['fecha_reg'].dt.date == fecha_objetivo].copy()

def cargar_descargas_a_bd(df):
    if df.empty: return
    engine = obtener_engine()
    
    mapeo_columnas = {
        'id_reg': 'id_reg', 'fecha_reg': 'fecha_reg', 'equipo': 'equipo', 'cod_equipo': 'cod_equipo',
        'nombre_operador': 'nombre_operador', 'horometro': 'horometro', 'cantidad_litros': 'cantidad_litros',
        'tipo_carga': 'tipo_carga', 'observaciones': 'observaciones', 'tanque': 'tanque',
        'propiedad': 'propiedad', 'labor': 'labor', 'usuario_reg': 'usuario_reg',
        'nombre_usuario': 'nombre_usuario', 'foto_horometro': 'foto_horometro', 'foto_contador': 'foto_contador'
    }
    
    df_sql = df.rename(columns=mapeo_columnas)[list(mapeo_columnas.values())]

    def upsert_descargas(table, conn, keys, data_iter):
        data = [dict(zip(keys, row)) for row in data_iter]
        insert_stmt = insert(table.table).values(data)
        update_dict = {c.name: c for c in insert_stmt.excluded if c.name != 'id_reg'}
        conn.execute(insert_stmt.on_conflict_do_update(index_elements=['id_reg'], set_=update_dict))

    print(f"🔄 [Descargas] Cargando {len(df_sql)} registros en PostgreSQL...")
    df_sql.to_sql(name="detalle_descargas", con=engine, if_exists='append', index=False, method=upsert_descargas, schema='datos_cosecha')


# =====================================================================
# 2. FLUJO: DETALLE MOVIMIENTOS (Estructura Corregida)
# =====================================================================
def extraer_y_filtrar_movimientos(fecha_busqueda_str):
    client = obtener_cliente_sheets()
    spreadsheet = client.open_by_key(SPREADSHEET_ID)
    sheet = spreadsheet.worksheet(WORKSHEET_MOVIMIENTOS)
    
    df = pd.DataFrame(sheet.get_all_records())
    if df.empty: return df
        
    df['fecha_reg'] = pd.to_datetime(df['fecha_reg'], format='mixed', errors='coerce')
    fecha_objetivo = pd.to_datetime(fecha_busqueda_str).date()
    return df[df['fecha_reg'].dt.date == fecha_objetivo].copy()

def cargar_movimientos_a_bd(df):
    if df.empty:
        print("⚠ No hay datos de movimientos para cargar en esta fecha.")
        return
    engine = obtener_engine()
    
    # Mapeo estricto según tu nuevo esquema
    mapeo_columnas = {
        'id_reg': 'id_reg',
        'fecha_reg': 'fecha_reg',
        'tanque_origen': 'tanque_origen',
        'responsable_tanque_origen': 'responsable_tanque_origen',
        'tanque_destino': 'tanque_destino',
        'responsable_tanque_destino': 'responsable_tanque_destino',
        'cantidad_litros': 'cantidad_litros',
        'foto_contador': 'foto_contador',
        'observaciones': 'observaciones',
        'usuario': 'usuario'
    }
    
    df_sql = df.rename(columns=mapeo_columnas)[list(mapeo_columnas.values())]

    def upsert_movimientos(table, conn, keys, data_iter):
        data = [dict(zip(keys, row)) for row in data_iter]
        insert_stmt = insert(table.table).values(data)
        update_dict = {c.name: c for c in insert_stmt.excluded if c.name != 'id_reg'}
        conn.execute(insert_stmt.on_conflict_do_update(index_elements=['id_reg'], set_=update_dict))

    print(f"🔄 [Movimientos] Cargando {len(df_sql)} registros en PostgreSQL...")
    try:
        df_sql.to_sql(
            name="detalle_movimientos",
            con=engine,
            if_exists='append',
            index=False,
            method=upsert_movimientos,
            schema='datos_cosecha'
        )
        print("✅ ¡Datos de movimientos cargados y actualizados correctamente!")
    except Exception as e:
        print(f"❌ Error al cargar los movimientos en la base de datos: {e}")


# =====================================================================
# 3. ORQUESTADOR GLOBAL DE COMBUSTIBLE
# =====================================================================
def ejecutar_sincronizacion_combustible(fecha_a_procesar):
    """Ejecuta de forma secuencial la descarga y carga de ambos procesos."""
    print(f"\n--- Sincronizando Combustible para la fecha: {fecha_a_procesar} ---")
    
    # Descargas
    df_desc = extraer_y_filtrar_descargas(fecha_a_procesar)
    cargar_descargas_a_bd(df_desc)
    
    print("- - - - - - - - - - - - - - - - - -")
    
    # Movimientos
    df_mov = extraer_y_filtrar_movimientos(fecha_a_procesar)
    cargar_movimientos_a_bd(df_mov)

In [11]:
fecha = '2026-07-11'
ejecutar_sincronizacion_combustible(fecha)


--- Sincronizando Combustible para la fecha: 2026-07-11 ---
🔄 [Descargas] Cargando 4 registros en PostgreSQL...
- - - - - - - - - - - - - - - - - -
⚠ No hay datos de movimientos para cargar en esta fecha.
